# Experiment 1: Structured S-CoT on Qwen2.5-1.5B (TPU)

This notebook performs Supervised Fine-Tuning (SFT) on the **Qwen2.5-1.5B-Instruct** model using structured reasoning traces. It is optimized for the **Google Colab TPU runtime** using the [Tunix](https://github.com/google/tunix) framework.

### 1. Environment Setup
We install Tunix and JAX with TPU support.

In [ ]:
%%capture
!pip install "google-tunix[prod]"
!pip install wandb huggingface_hub gcsfs datasets evaluate tqdm jsonlines python-dotenv

### 2. Verify TPU Access

In [ ]:
import jax
print("TPU devices:", jax.devices())

### 3. Clone Repository & Load Data

In [ ]:
import os
REPO_URL = "https://github.com/TinevimboMusingadi/scot-reasoning-.git"
REPO_DIR = "scot-reasoning-"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

DATA_PATH = os.path.join(REPO_DIR, "data/full_run/scot_traces.jsonl")
print(f"Data path: {DATA_PATH}")

### 4. Initialize Model and Tokenizer (Tunix)
We use Tunix's JAX-native model definitions for Qwen2.

In [ ]:
import jax.numpy as jnp
from huggingface_hub import snapshot_download
from tunix.models.qwen2 import model as qwen_lib
from tunix.models.qwen2 import params_safetensors as qwen_params
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MESH = [(1, 8), ("fsdp", "tp")] # Assuming v2-8 or v3-8
mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * 2)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
SCOT_TOKENS = [
    "<reasoning>", "</reasoning>",
    "<meta_reasoning>", "</meta_reasoning>",
    "<abduction>", "</abduction>",
    "<decompose>", "</decompose>",
    "<deduction>", "</deduction>",
    "<induction>", "</induction>",
    "<analogy>", "</analogy>",
    "<causal>", "</causal>",
    "<answer>", "</answer>",
]
tokenizer.add_special_tokens({"additional_special_tokens": SCOT_TOKENS})

model_path = snapshot_download(repo_id=MODEL_ID, ignore_patterns=["*.pth"])
config = qwen_lib.ModelConfig.qwen2_5_1_5b()
config.vocab_size = len(tokenizer)

with mesh:
    model = qwen_params.create_model_from_safe_tensors(
        model_path, config, mesh, dtype=jnp.bfloat16
    )

### 5. Apply QLoRA (Qwix)

In [ ]:
import qwix
lora_provider = qwix.LoraProvider(
    module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
    rank=16,
    alpha=32,
    weight_qtype="nf4",
    tile_size=128,
)
model_input = model.get_model_input()
lora_model  = qwix.apply_lora_to_model(model, lora_provider, **model_input)

### 6. Training with PeftTrainer

In [ ]:
from tunix.sft import peft_trainer, training_config
import optax
import jsonlines, numpy as np

def build_prompt(problem, trace):
    return f"<|im_start|>user\n{problem}<|im_end|>\n<|im_start|>assistant\n{trace}<|im_end|>"

def load_dataset(path, tokenizer, max_seq_len=2048):
    examples = []
    with jsonlines.open(path) as reader:
        for row in reader:
            text = build_prompt(row["problem"], row["scot_trace"])
            tokens = tokenizer.encode(text)[:max_seq_len]
            pad_len = max_seq_len - len(tokens)
            input_tokens = tokens + [tokenizer.pad_id()] * pad_len
            input_mask   = [1] * len(tokens) + [0] * pad_len
            examples.append({
                "input_tokens": np.array(input_tokens, dtype=np.int32),
                "input_mask":   np.array(input_mask,   dtype=np.int32),
            })
    return examples

train_data = load_dataset(DATA_PATH, tokenizer)
t_config = training_config.TrainingConfig(eval_every_n_steps=100, max_steps=500)
trainer = peft_trainer.PeftTrainer(
    model=lora_model,
    optimizer=optax.adamw(learning_rate=2e-5),
    training_config=t_config,
)

trainer.with_gen_model_input_fn(lambda x: {"input_tokens": x["input_tokens"], "input_mask": x["input_mask"]})
trainer.train(train_ds=train_data)

### 7. Inference Test

In [ ]:
# Note: Inference in Tunix typically uses the Sampler API
from tunix.generate import sampler as sampler_lib

sampler = sampler_lib.Sampler(
    transformer=lora_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=512,
        num_layers=config.num_hidden_layers,
        num_kv_heads=config.num_key_value_heads,
        head_dim=config.hidden_size // config.num_attention_heads,
    ),
)

test_prompt = build_prompt("If a baker has 10 loaves and sells 3, then bakes 5 more, how many does he have?", "")
output = sampler(input_strings=[test_prompt], max_generation_steps=512)
print(output[0])